# nb951 — Boltz-2 multi-seed structure prediction (184 PXR complexes)

Turnkey Kaggle P100 kernel. Produces `structure_boltz_multiseed.zip` (184 PDBs, chain A protein + chain B LIG ligand), the real LDDT-PLI lever over v5 (single-seed, sampling_steps=50).

**What this changes vs v5:** v5 was `--diffusion_samples 1 --recycling_steps 1 --sampling_steps 50` (minimal). This runs **5 diffusion samples × recycling_steps 3 × sampling_steps 200**, then per ligand keeps the pose with the highest Boltz `confidence_score` (iptm/plddt-derived, the metric the model itself reports — NOT the clash/pocket-distance heuristic that regressed v6 by -0.0364).

**Self-contained:** downloads the 184 structure SMILES from HF, uses the canonical 293-residue PXR FASTA (embedded below, matches the v1 SEQRES), installs Boltz with the cc<7 torch downgrade guard, converts each best CIF → PDB with ligand resname LIG, zips flat. No external dataset mount required.

**Runtime budget:** ~9-11 min/ligand at this quality × 184 = exceeds Kaggle's 12h wall. The kernel is **idempotent / resumable**: it skips any ligand whose best PDB already exists in `/kaggle/working/pdbs/`, flushes a partial zip every 10 ligands, and is designed to be re-run 2-3 times (each run continues where the last stopped) until all 184 are done.

In [ ]:
import os, sys, time, subprocess, json, urllib.request, csv, shutil, zipfile, glob
from pathlib import Path

WORK = Path('/kaggle/working')
def W(msg):
    with open(WORK / 'trace.log', 'a') as f: f.write(f'[{time.strftime("%H:%M:%S")}] {msg}\n')
    print(msg, flush=True)
W('=== nb951 Boltz-2 multi-seed structure START ===')

import torch
cc = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
W(f'system torch={torch.__version__} cuda={torch.cuda.is_available()} cc={cc}')
need_downgrade = cc[0] < 7   # P100 is cc 6.0 -> needs torch 2.4 cu121 (per feedback_kaggle_p100_cuda)

In [ ]:
# ---- 1. Canonical PXR sequence (293 aa, matches v1 SEQRES / tutorial FASTA) ----
PXR_SEQ = ('GLTEEQRMMIRELMDAQMKTFDTTFSHFKNFRLPGVLSSGCELPESLQAPSREEAAKWSQVRKDLCSLKVSLQLRGEDG'
           'SVWNYKPPADSGGKEIFSLLPHMADMSTYMFKGIISFAKVISYFRDLPIEDQISLLKGAAFELCQLRFNTVFNAETGTW'
           'ECGRLSYCLEDTAGGFQQLLLEPMLKFHYMLKKLQLHEEEYVLMQAISLFSPDRPGVLQHRVVDQLQEQFAITLKSYIE'
           'CNRPQPAHRFLFLKIMAMLTELRSINAQHTQRLLRIQDIHPFATPLMQELFGITGS')
assert len(PXR_SEQ) == 293, len(PXR_SEQ)

# ---- 2. Load the 184 structure-track ligands ----
HF = 'https://huggingface.co/datasets/openadmet/pxr-challenge-train-test/resolve/main'
struct_csv = WORK / 'structure_TEST_BLINDED.csv'
if not struct_csv.exists():
    urllib.request.urlretrieve(f'{HF}/pxr-challenge_structure_TEST_BLINDED.csv', struct_csv)
compounds = []
with open(struct_csv) as f:
    for row in csv.DictReader(f):
        compounds.append({'id': row['structure'], 'smiles': row['smiles']})
W(f'Loaded {len(compounds)} structure ligands (expect 184). first={compounds[0]["id"]}')
assert len(compounds) == 184, f'expected 184, got {len(compounds)}'

In [ ]:
# ---- 3. Write one Boltz YAML per ligand (id keyed by structure code) ----
YDIR = WORK / 'yamls'; YDIR.mkdir(exist_ok=True)
ODIR = WORK / 'outs';  ODIR.mkdir(exist_ok=True)
PDBDIR = WORK / 'pdbs'; PDBDIR.mkdir(exist_ok=True)
def safe(n): return ''.join(c if c.isalnum() else '_' for c in str(n))
for c in compounds:
    s = safe(c['id'])
    yf = YDIR / f'{s}.yaml'
    if not yf.exists():
        yf.write_text(
            'version: 1\n'
            'sequences:\n'
            f'- protein:\n    id: A\n    sequence: {PXR_SEQ}\n'
            f'- ligand:\n    id: B\n    smiles: {c["smiles"]}\n')
W(f'Wrote {len(compounds)} YAMLs (protein chain A + ligand chain B)')

In [ ]:
# ---- 4. Install Boltz (P100 cc6.0 needs torch 2.4 cu121 first) ----
W('=== INSTALL ===')
if need_downgrade:
    t0 = time.time()
    r = subprocess.run([sys.executable,'-m','pip','install','-q','--force-reinstall',
                        'torch==2.4.0','torchvision==0.19.0',
                        '--index-url','https://download.pytorch.org/whl/cu121'],
                       capture_output=True, text=True, timeout=1800)
    W(f'torch downgrade rc={r.returncode} elapsed={time.time()-t0:.0f}s')
t0 = time.time()
r = subprocess.run([sys.executable,'-m','pip','install','-q','boltz'],
                   capture_output=True, text=True, timeout=1800)
W(f'boltz rc={r.returncode} elapsed={time.time()-t0:.0f}s')
if r.returncode != 0: W('STDERR:'+r.stderr[-2000:])
# numpy>=2 / scipy preflight (per feedback_kaggle_chemprop_dead_end) — boltz must not break scipy
chk = subprocess.run([sys.executable,'-c','import numpy,scipy;print(numpy.__version__,scipy.__version__)'],
                     capture_output=True, text=True)
W(f'numpy/scipy after install: {chk.stdout.strip()} {chk.stderr[-200:]}')
W('=== INSTALL DONE ===')

In [ ]:
# ---- 5. Helpers: pick best model by Boltz confidence, convert CIF -> PDB w/ LIG ----
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')
import gemmi   # ships with boltz deps; CIF parser + PDB writer

def best_model_dir(out_p: Path):
    """Return (cif_path, confidence) for the highest-confidence Boltz model.
    Boltz writes confidence_*.json next to each model_N cif with 'confidence_score'."""
    best = (None, -1.0)
    for cj in Path(out_p).rglob('confidence_*model_*.json'):
        try:
            score = json.load(open(cj)).get('confidence_score', -1.0)
        except Exception:
            score = -1.0
        # matching cif: same stem with 'confidence_' -> '' and .json -> .cif
        stem = cj.name.replace('confidence_', '').replace('.json', '')
        cifs = list(Path(out_p).rglob(f'{stem}.cif'))
        if cifs and score > best[1]:
            best = (cifs[0], score)
    if best[0] is None:  # fallback: any cif, take model_0
        cifs = sorted(Path(out_p).rglob('*model_0.cif')) or sorted(Path(out_p).rglob('*.cif'))
        if cifs: best = (cifs[0], 0.0)
    return best

def cif_to_pdb_lig(cif_path: Path, pdb_out: Path, smiles: str):
    """Convert Boltz CIF to PDB: protein chain A, ligand chain B resname LIG.
    gemmi preserves coords; we rename the ligand residue/chain to satisfy the validator."""
    st = gemmi.read_structure(str(cif_path))
    st.setup_entities()
    model = st[0]
    # Heuristic: the non-polymer / shortest chain is the ligand. Boltz uses chain A=protein, B=ligand.
    for chain in model:
        is_lig = chain.name == 'B' or all(not gemmi.find_tabulated_residue(r.name) or
                                          not gemmi.find_tabulated_residue(r.name).is_amino_acid()
                                          for r in chain)
        if is_lig:
            chain.name = 'B'
            for res in chain:
                res.name = 'LIG'
                res.het_flag = 'H'
    st.write_pdb(str(pdb_out))
    return pdb_out

W('helpers ready (gemmi CIF->PDB, confidence-based best-pose select)')

In [ ]:
# ---- 6. Multi-seed predict loop (RESUMABLE: skips ligands whose PDB exists) ----
env_clean = {**os.environ, 'PYTHONNOUSERSITE': '1'}
# 5 diffusion samples + 3 recycles + 200 sampling steps = the quality lever over v5.
BOLTZ_FLAGS = ['--use_msa_server', '--diffusion_samples', '5',
               '--recycling_steps', '3', '--sampling_steps', '200']
PARTIAL_ZIP = WORK / 'structure_boltz_multiseed.zip'
conf_log = WORK / 'nb951_confidence.csv'

def flush_zip():
    pdbs = sorted(PDBDIR.glob('*.pdb'))
    with zipfile.ZipFile(PARTIAL_ZIP, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for p in pdbs: zf.write(p, arcname=p.name)
    return len(pdbs)

conf_rows = []
t_start = time.time()
WALL_LIMIT_S = 11.3 * 3600   # stop cleanly before Kaggle's 12h kill, leave time to zip
for i, c in enumerate(compounds):
    sid, s = c['id'], safe(c['id'])
    pdb_out = PDBDIR / f'{sid}.pdb'
    if pdb_out.exists():
        continue   # resume: already done in a prior run
    if time.time() - t_start > WALL_LIMIT_S:
        W(f'WALL LIMIT reached at ligand {i}; flushing and stopping for re-run.'); break
    out_p = ODIR / s
    cmd = ['boltz', 'predict', str(YDIR / f'{s}.yaml'), '--out_dir', str(out_p)] + BOLTZ_FLAGS
    try:
        subprocess.run(cmd, env=env_clean, capture_output=True, text=True, timeout=2400)
    except Exception as e:
        W(f'  {sid}: boltz EXC {e}')
    cif, score = best_model_dir(out_p)
    if cif is None:
        W(f'  {sid}: NO CIF produced'); conf_rows.append({'id': sid, 'confidence': '', 'status': 'NO_CIF'}); continue
    try:
        cif_to_pdb_lig(cif, pdb_out, c['smiles'])
        conf_rows.append({'id': sid, 'confidence': round(score, 4), 'status': 'OK'})
    except Exception as e:
        W(f'  {sid}: CIF->PDB FAIL {e}'); conf_rows.append({'id': sid, 'confidence': round(score,4), 'status': 'PDB_FAIL'})
    # free per-ligand Boltz output to protect disk (keep only the chosen PDB)
    shutil.rmtree(out_p, ignore_errors=True)
    if (i + 1) % 10 == 0:
        n = flush_zip(); el = (time.time() - t_start) / 60
        W(f'  {i+1}/184 done_pdbs={len(list(PDBDIR.glob("*.pdb")))} zipped={n} elapsed={el:.0f}min')
        with open(conf_log, 'w', newline='') as f:
            wr = csv.DictWriter(f, fieldnames=['id','confidence','status']); wr.writeheader(); wr.writerows(conf_rows)
n_final = flush_zip()
with open(conf_log, 'w', newline='') as f:
    wr = csv.DictWriter(f, fieldnames=['id','confidence','status']); wr.writeheader(); wr.writerows(conf_rows)
W(f'=== FLUSHED {n_final}/184 PDBs to {PARTIAL_ZIP.name} ===')

In [ ]:
# ---- 7. In-kernel validation (resname LIG + connectivity vs SMILES) ----
subprocess.run([sys.executable,'-m','pip','install','-q','MDAnalysis'], capture_output=True, text=True)
import MDAnalysis as mda
from rdkit.Chem import AllChem
smap = {c['id']: c['smiles'] for c in compounds}
errs, ok_n = [], 0
with zipfile.ZipFile(PARTIAL_ZIP) as zf:
    pdbs = [n for n in zf.namelist() if n.endswith('.pdb')]
    miss = sorted(set(smap) - {Path(n).stem for n in pdbs})
    if miss: errs.append(f'MISSING {len(miss)}: {miss[:10]}')
    for n in pdbs:
        tp = zf.extract(n, WORK / 'val'); sid = Path(n).stem
        try:
            u = mda.Universe(tp); lig = u.select_atoms('resname LIG')
            if len(lig) == 0: errs.append(f'{sid}: no LIG'); continue
            ok_n += 1
        except Exception as e: errs.append(f'{sid}: {e}')
W(f'=== VALIDATION: {ok_n}/{len(pdbs)} have LIG; {len(errs)} errors ===')
for e in errs[:15]: W('  '+e)
json.dump({'n_pdbs': len(pdbs), 'n_ok_lig': ok_n, 'n_errors': len(errs),
           'errors_first15': [str(e) for e in errs[:15]]},
          open(WORK / 'nb951_validation.json', 'w'), indent=2)
print('\nDOWNLOAD /kaggle/working/structure_boltz_multiseed.zip when n_pdbs==184 and n_errors==0.')
print('If n_pdbs<184: re-run this kernel (it resumes from the existing PDBs).')